# Vesuvius Challenge 2025 - TPU Training (PyTorch XLA)

## Model: 6-stage ResEncUNet3D with scSE + Topology-Aware Losses

**Target Hardware:** Kaggle TPU v5e-8 (8 TPU cores)

**Framework:** PyTorch + torch_xla

### Architecture (Same as GPU version):
| Component | Specification |
|-----------|---------------|
| Model | 6-stage ResEncUNet3D |
| Features | [32, 64, 128, 256, 320, 320] |
| Blocks | [1, 3, 4, 6, 6, 6] |
| Attention | scSE (Spatial-Channel Squeeze-Excitation) |
| Normalization | GroupNorm |
| Deep Supervision | Yes (3 heads) |
| Patch Size | 192x192x192 |

### Loss Schedule:
- Epochs 0-299: Dice + BCE
- Epochs 300-599: Add SkeletonRecall
- Epochs 600-1200: Add soft-clDice

### References:
- [PyTorch XLA on Kaggle](https://github.com/pytorch/xla/blob/master/contrib/kaggle/pytorch-xla-2-0-on-kaggle.ipynb)
- [Distributed PyTorch XLA](https://github.com/pytorch/xla/blob/master/contrib/kaggle/distributed-pytorch-xla-basics-with-pjrt.ipynb)

In [ ]:
# =============================================================================
# CELL 1: SETUP & INSTALLATION
# =============================================================================

from IPython.display import clear_output

# Install PyTorch XLA for TPU
!pip install torch~=2.6.0 torch_xla[tpu]~=2.6.0 -f https://storage.googleapis.com/libtpu-releases/index.html -q

# Install tifffile for loading TIFF volumes
!pip install tifffile imagecodecs -q

clear_output()
print("Dependencies installed")

In [ ]:
# =============================================================================
# CELL 2: IMPORTS & TPU CONFIGURATION
# =============================================================================

import os
import gc
import json
import random
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass

import numpy as np
import pandas as pd
import tifffile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, DistributedSampler

# PyTorch XLA imports
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.runtime as xr  # NEW: PJRT runtime API (replaces deprecated xm.xrt_* functions)
import torch_xla.distributed.xla_multiprocessing as xmp
import torch_xla.distributed.parallel_loader as pl

warnings.filterwarnings('ignore')

# Verify TPU availability
print(f"PyTorch version: {torch.__version__}")
print(f"torch_xla version: {torch_xla.__version__}")
print(f"XLA device: {xm.xla_device()}")
print(f"World size: {xr.world_size()}")  # NEW: Use xr.world_size() instead of xm.xrt_world_size()

In [ ]:
# =============================================================================
# CELL 3: CONFIGURATION
# =============================================================================

@dataclass
class Config:
    # Data paths (Kaggle)
    DATA_ROOT: str = "/kaggle/input/3d-volume-training-data"
    CHECKPOINT_DIR: str = "/kaggle/working/checkpoints"
    
    # Model architecture (SAME AS GPU VERSION)
    PATCH_SIZE: Tuple[int, int, int] = (192, 192, 192)
    FEATURES: List[int] = None
    N_BLOCKS: List[int] = None
    USE_SCSE: bool = True
    USE_DEEP_SUPERVISION: bool = True
    
    # Training - TPU specific
    EPOCHS: int = 1200
    BATCH_SIZE_PER_DEVICE: int = 1  # Per TPU core
    NUM_TPU_CORES: int = 8  # TPU v5e-8
    
    # Loss weights (SAME AS GPU VERSION)
    DICE_WEIGHT: float = 0.3
    BCE_WEIGHT: float = 0.2
    SKELETON_WEIGHT: float = 0.25
    CLDICE_WEIGHT: float = 0.25
    
    # Loss scheduling (SAME AS GPU VERSION)
    SKELETON_START_EPOCH: int = 300
    SKELETON_WARMUP_EPOCHS: int = 300
    CLDICE_START_EPOCH: int = 600
    CLDICE_WARMUP_EPOCHS: int = 300
    
    # Optimizer
    LEARNING_RATE: float = 0.01
    MOMENTUM: float = 0.99
    WEIGHT_DECAY: float = 3e-5
    GRADIENT_CLIP: float = 12.0
    
    # Data
    DATA_FRACTION: float = 1.0
    PATCHES_PER_VOLUME: int = 8
    PRELOAD_ALL: bool = True
    
    # Training control
    VALIDATE_EVERY: int = 5
    SAVE_EVERY: int = 10
    
    # Random seed
    SEED: int = 42
    
    def __post_init__(self):
        if self.FEATURES is None:
            self.FEATURES = [32, 64, 128, 256, 320, 320]
        if self.N_BLOCKS is None:
            self.N_BLOCKS = [1, 3, 4, 6, 6, 6]
        os.makedirs(self.CHECKPOINT_DIR, exist_ok=True)


cfg = Config()
cfg.__post_init__()

print("="*60)
print("TPU CONFIGURATION")
print("="*60)
print(f"Data root: {cfg.DATA_ROOT}")
print(f"Patch size: {cfg.PATCH_SIZE}")
print(f"Features: {cfg.FEATURES}")
print(f"Blocks: {cfg.N_BLOCKS}")
print(f"Batch size per device: {cfg.BATCH_SIZE_PER_DEVICE}")
print(f"Total batch size: {cfg.BATCH_SIZE_PER_DEVICE * cfg.NUM_TPU_CORES}")
print(f"Epochs: {cfg.EPOCHS}")
print("="*60)

In [ ]:
# =============================================================================
# CELL 4: MODEL (6-stage ResEncUNet3D + scSE + GroupNorm)
# SAME AS GPU VERSION - Pure PyTorch works on XLA
# =============================================================================

class ConvBlock(nn.Module):
    """3D convolution block with GroupNorm."""
    def __init__(self, in_ch, out_ch, num_groups=8):
        super().__init__()
        num_groups = min(num_groups, out_ch)
        while out_ch % num_groups != 0:
            num_groups -= 1
        
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(num_groups, out_ch),
            nn.LeakyReLU(0.01, inplace=True),
        )
    
    def forward(self, x):
        return self.conv(x)


class ResBlock(nn.Module):
    """Residual block with N convolutions."""
    def __init__(self, channels, n_convs=2):
        super().__init__()
        self.blocks = nn.Sequential(
            *[ConvBlock(channels, channels) for _ in range(n_convs)]
        )
    
    def forward(self, x):
        return x + self.blocks(x)


class scSEBlock(nn.Module):
    """Concurrent Spatial and Channel Squeeze-and-Excitation."""
    def __init__(self, channels, reduction=2):
        super().__init__()
        self.cse_pool = nn.AdaptiveAvgPool3d(1)
        self.cse_fc1 = nn.Linear(channels, channels // reduction)
        self.cse_fc2 = nn.Linear(channels // reduction, channels)
        self.sse_conv = nn.Conv3d(channels, 1, 1)
    
    def forward(self, x):
        b, c, d, h, w = x.shape
        cse = self.cse_pool(x).view(b, c)
        cse = F.relu(self.cse_fc1(cse))
        cse = torch.sigmoid(self.cse_fc2(cse)).view(b, c, 1, 1, 1)
        sse = torch.sigmoid(self.sse_conv(x))
        return x * cse + x * sse


class ResEncUNet3D(nn.Module):
    """
    6-stage Residual Encoder U-Net with scSE attention and GroupNorm.
    Same architecture as GPU version.
    """
    
    def __init__(
        self,
        in_ch: int = 1,
        out_ch: int = 1,
        features: List[int] = None,
        n_blocks: List[int] = None,
        use_scse: bool = True,
        use_deep_supervision: bool = True,
    ):
        super().__init__()
        
        if features is None:
            features = [32, 64, 128, 256, 320, 320]
        if n_blocks is None:
            n_blocks = [1, 3, 4, 6, 6, 6]
        
        self.features = features
        self.use_scse = use_scse
        self.use_deep_supervision = use_deep_supervision
        self.n_stages = len(features)
        
        # Encoder
        self.encoders = nn.ModuleList()
        self.attentions = nn.ModuleList()
        self.pools = nn.ModuleList()
        
        for i, (feat, nb) in enumerate(zip(features, n_blocks)):
            in_channels = in_ch if i == 0 else features[i - 1]
            encoder = nn.Sequential(
                ConvBlock(in_channels, feat),
                *[ResBlock(feat, n_convs=2) for _ in range(nb)]
            )
            self.encoders.append(encoder)
            
            if use_scse:
                self.attentions.append(scSEBlock(feat))
            else:
                self.attentions.append(nn.Identity())
            
            if i < len(features) - 1:
                self.pools.append(nn.Conv3d(feat, feat, kernel_size=2, stride=2))
        
        # Decoder
        self.ups = nn.ModuleList()
        self.dec_convs = nn.ModuleList()
        
        for i in range(len(features) - 2, -1, -1):
            up_feat = features[i + 1]
            out_feat = features[i]
            self.ups.append(nn.ConvTranspose3d(up_feat, out_feat, kernel_size=2, stride=2))
            self.dec_convs.append(ConvBlock(out_feat * 2, out_feat))
        
        self.final = nn.Conv3d(features[0], out_ch, 1)
        
        # Deep supervision heads
        if use_deep_supervision:
            self.ds_heads = nn.ModuleList()
            n_ds_outputs = min(3, len(features) - 1)
            for i in range(n_ds_outputs):
                ds_in_channels = features[-(i + 2)]
                self.ds_heads.append(nn.Conv3d(ds_in_channels, out_ch, 1))
    
    def forward(self, x):
        enc_features = []
        
        for i, (enc, att) in enumerate(zip(self.encoders, self.attentions)):
            x = enc(x)
            x = att(x)
            enc_features.append(x)
            if i < len(self.pools):
                x = self.pools[i](x)
        
        enc_features = enc_features[::-1]
        x = enc_features[0]
        
        ds_outputs = []
        
        for i, (up, dec) in enumerate(zip(self.ups, self.dec_convs)):
            x = up(x)
            skip = enc_features[i + 1]
            
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            
            x = torch.cat([x, skip], dim=1)
            x = dec(x)
            
            if self.use_deep_supervision and self.training and i < len(self.ds_heads):
                ds_outputs.append(self.ds_heads[i](x))
        
        out = self.final(x)
        
        if self.use_deep_supervision and self.training:
            return {'output': out, 'deep': ds_outputs}
        return out


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Model defined (same as GPU version)")

In [ ]:
# =============================================================================
# CELL 5: LOSS FUNCTIONS (Same as GPU version)
# =============================================================================

class DiceLoss(nn.Module):
    """Dice loss with ignore mask support."""
    def __init__(self, smooth: float = 1e-5):
        super().__init__()
        self.smooth = smooth
    
    def forward(self, pred, target, mask=None):
        pred = torch.sigmoid(pred)
        
        if mask is not None:
            pred = pred * (1 - mask)
            target = target * (1 - mask)
        
        intersection = (pred * target).sum()
        union = pred.sum() + target.sum()
        
        dice = (2 * intersection + self.smooth) / (union + self.smooth)
        return 1 - dice


class BCEWithMask(nn.Module):
    """BCE loss with ignore mask support."""
    def forward(self, pred, target, mask=None):
        if mask is not None:
            valid = 1 - mask
            loss = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
            loss = (loss * valid).sum() / (valid.sum() + 1e-8)
        else:
            loss = F.binary_cross_entropy_with_logits(pred, target)
        return loss


def soft_skeletonize(x, num_iter=40):
    """Soft skeletonization using iterative min-max pooling."""
    for _ in range(num_iter):
        min_pool = -F.max_pool3d(-x, 3, stride=1, padding=1)
        max_min_pool = F.max_pool3d(min_pool, 3, stride=1, padding=1)
        x = F.relu(x - max_min_pool)
    return x


class SoftClDiceLoss(nn.Module):
    """Soft clDice (centerline Dice) loss."""
    def __init__(self, smooth: float = 1e-5, num_iter: int = 10):
        super().__init__()
        self.smooth = smooth
        self.num_iter = num_iter
    
    def forward(self, pred, target, mask=None):
        pred = torch.sigmoid(pred)
        
        if mask is not None:
            pred = pred * (1 - mask)
            target = target * (1 - mask)
        
        skel_pred = soft_skeletonize(pred, self.num_iter)
        skel_target = soft_skeletonize(target, self.num_iter)
        
        tprec = ((skel_pred * target).sum() + self.smooth) / (skel_pred.sum() + self.smooth)
        tsens = ((skel_target * pred).sum() + self.smooth) / (skel_target.sum() + self.smooth)
        cldice = 2 * tprec * tsens / (tprec + tsens + self.smooth)
        
        return 1 - cldice


class SkeletonRecallLoss(nn.Module):
    """Skeleton Recall Loss."""
    def __init__(self, smooth: float = 1e-5, tube_radius: int = 2):
        super().__init__()
        self.smooth = smooth
        self.tube_radius = tube_radius
    
    def forward(self, pred, target, mask=None):
        pred_sigmoid = torch.sigmoid(pred)
        
        if mask is not None:
            pred_sigmoid = pred_sigmoid * (1 - mask)
            target = target * (1 - mask)
        
        target_skel = soft_skeletonize(target, num_iter=10)
        
        if self.tube_radius > 0:
            target_tube = F.max_pool3d(
                target_skel,
                kernel_size=2 * self.tube_radius + 1,
                stride=1,
                padding=self.tube_radius
            )
        else:
            target_tube = target_skel
        
        recall = (pred_sigmoid * target_tube).sum() / (target_tube.sum() + self.smooth)
        return 1 - recall


class CombinedLoss(nn.Module):
    """Combined loss with scheduling for topology losses."""
    def __init__(
        self,
        dice_weight: float = 0.3,
        bce_weight: float = 0.2,
        skeleton_weight: float = 0.25,
        cldice_weight: float = 0.25,
        skeleton_start: int = 300,
        skeleton_warmup: int = 300,
        cldice_start: int = 600,
        cldice_warmup: int = 300,
        ds_weights: List[float] = None,
    ):
        super().__init__()
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight
        self.skeleton_weight = skeleton_weight
        self.cldice_weight = cldice_weight
        
        self.skeleton_start = skeleton_start
        self.skeleton_warmup = skeleton_warmup
        self.cldice_start = cldice_start
        self.cldice_warmup = cldice_warmup
        
        if ds_weights is None:
            ds_weights = [0.5, 0.25, 0.125]
        self.ds_weights = ds_weights
        
        self.dice_loss = DiceLoss()
        self.bce_loss = BCEWithMask()
        self.skeleton_loss = SkeletonRecallLoss()
        self.cldice_loss = SoftClDiceLoss()
        
        # Track current epoch (updated externally)
        self.current_epoch = 0
    
    def _get_scale(self, epoch, start, warmup):
        if epoch < start:
            return 0.0
        elif epoch < start + warmup:
            return (epoch - start) / warmup
        return 1.0
    
    def forward(self, output, target, ignore_mask):
        epoch = self.current_epoch
        
        if isinstance(output, dict):
            pred = output['output']
            deep_outputs = output.get('deep', [])
        else:
            pred = output
            deep_outputs = []
        
        skel_scale = self._get_scale(epoch, self.skeleton_start, self.skeleton_warmup)
        cldice_scale = self._get_scale(epoch, self.cldice_start, self.cldice_warmup)
        
        dice = self.dice_loss(pred, target, ignore_mask)
        bce = self.bce_loss(pred, target, ignore_mask)
        
        if skel_scale > 0:
            skel = self.skeleton_loss(pred, target, ignore_mask)
        else:
            skel = torch.tensor(0.0, device=pred.device)
        
        if cldice_scale > 0:
            cldice = self.cldice_loss(pred, target, ignore_mask)
        else:
            cldice = torch.tensor(0.0, device=pred.device)
        
        total = (
            self.dice_weight * dice +
            self.bce_weight * bce +
            self.skeleton_weight * skel_scale * skel +
            self.cldice_weight * cldice_scale * cldice
        )
        
        # Deep supervision losses
        for i, ds_out in enumerate(deep_outputs):
            if i >= len(self.ds_weights):
                break
            ds_target = F.interpolate(target, size=ds_out.shape[2:], mode='nearest')
            ds_mask = F.interpolate(ignore_mask, size=ds_out.shape[2:], mode='nearest')
            ds_dice = self.dice_loss(ds_out, ds_target, ds_mask)
            total = total + self.ds_weights[i] * ds_dice
        
        return total, dice.item()


print("Loss functions defined")
print(f"Loss scheduling:")
print(f"  Epochs 0-{cfg.SKELETON_START_EPOCH}: Dice + BCE only")
print(f"  Epochs {cfg.SKELETON_START_EPOCH}-{cfg.CLDICE_START_EPOCH}: Add SkeletonRecall")
print(f"  Epochs {cfg.CLDICE_START_EPOCH}-{cfg.EPOCHS}: Add soft-clDice")

In [ ]:
# =============================================================================
# CELL 6: DATASET (Same as GPU version)
# =============================================================================

def load_volume_tiff(path: str) -> np.ndarray:
    """Load 3D TIF volume."""
    return tifffile.imread(str(path))


class VesuviusDataset(Dataset):
    """
    Dataset for Vesuvius 3D volumes from TIFF files.
    Preloads all data to RAM for efficient training.
    """
    
    def __init__(
        self,
        data_root: str,
        patch_size: Tuple[int, int, int] = (192, 192, 192),
        is_train: bool = True,
        data_fraction: float = 1.0,
        patches_per_volume: int = 8,
        preload: bool = True,
    ):
        self.data_root = Path(data_root)
        self.patch_size = patch_size
        self.is_train = is_train
        self.patches_per_volume = patches_per_volume
        
        # Find data files
        train_csv = self.data_root / "train.csv"
        self.images_dir = self.data_root / "train_images"
        self.labels_dir = self.data_root / "train_labels"
        
        # Load CSV and filter valid IDs
        df = pd.read_csv(train_csv)
        valid_ids = []
        for idx in df['id'].values:
            img_path = self.images_dir / f"{idx}.tif"
            lbl_path = self.labels_dir / f"{idx}.tif"
            if img_path.exists() and lbl_path.exists():
                valid_ids.append(idx)
        
        # Apply data fraction
        if data_fraction < 1.0:
            n = max(1, int(len(valid_ids) * data_fraction))
            random.shuffle(valid_ids)
            valid_ids = valid_ids[:n]
        
        self.volume_ids = valid_ids
        
        # Storage for preloaded data
        self.volumes = {}
        self.fg_coords = {}
        
        # Preload all data to RAM
        if preload:
            print(f"Preloading {len(self.volume_ids)} volumes to RAM...")
            for vol_id in tqdm(self.volume_ids, desc="Loading"):
                img = load_volume_tiff(self.images_dir / f"{vol_id}.tif").astype(np.float32)
                lbl = load_volume_tiff(self.labels_dir / f"{vol_id}.tif").astype(np.uint8)
                
                # Normalize
                img = (img - img.mean()) / (img.std() + 1e-8)
                
                self.volumes[vol_id] = (img, lbl)
                
                # Cache foreground coordinates
                fg = np.argwhere(lbl == 1)
                if len(fg) > 10000:
                    fg = fg[np.random.choice(len(fg), 10000, replace=False)]
                self.fg_coords[vol_id] = fg if len(fg) > 0 else None
            
            sample_vol = next(iter(self.volumes.values()))
            vol_size_mb = (sample_vol[0].nbytes + sample_vol[1].nbytes) / 1e6
            total_gb = vol_size_mb * len(self.volumes) / 1000
            print(f"Loaded {len(self.volumes)} volumes ({total_gb:.1f} GB in RAM)")
        
        print(f"Dataset ready: {len(self)} samples")
    
    def __len__(self):
        return len(self.volume_ids) * self.patches_per_volume
    
    def __getitem__(self, idx):
        vol_idx = idx // self.patches_per_volume
        vol_id = self.volume_ids[vol_idx]
        
        image, label = self.volumes[vol_id]
        
        d, h, w = image.shape
        pd, ph, pw = self.patch_size
        
        # Pad if needed
        if d < pd or h < ph or w < pw:
            pad_d, pad_h, pad_w = max(0, pd-d), max(0, ph-h), max(0, pw-w)
            image = np.pad(image, ((0,pad_d),(0,pad_h),(0,pad_w)), mode='reflect')
            label = np.pad(label, ((0,pad_d),(0,pad_h),(0,pad_w)), mode='constant', constant_values=2)
            d, h, w = image.shape
        
        # Foreground-centered sampling (60% of time)
        fg = self.fg_coords.get(vol_id)
        if self.is_train and random.random() < 0.6 and fg is not None and len(fg) > 0:
            c = fg[random.randint(0, len(fg)-1)]
            z = max(0, min(c[0] - pd//2, d - pd))
            y = max(0, min(c[1] - ph//2, h - ph))
            x = max(0, min(c[2] - pw//2, w - pw))
        else:
            z = random.randint(0, max(0, d - pd))
            y = random.randint(0, max(0, h - ph))
            x = random.randint(0, max(0, w - pw))
        
        img_patch = image[z:z+pd, y:y+ph, x:x+pw].copy()
        lbl_patch = label[z:z+pd, y:y+ph, x:x+pw].copy()
        
        # Augmentation
        if self.is_train:
            for ax in range(3):
                if random.random() > 0.5:
                    img_patch = np.flip(img_patch, ax)
                    lbl_patch = np.flip(lbl_patch, ax)
            
            k = random.randint(0, 3)
            if k:
                img_patch = np.rot90(img_patch, k, (1,2))
                lbl_patch = np.rot90(lbl_patch, k, (1,2))
            
            img_patch = np.ascontiguousarray(img_patch)
            lbl_patch = np.ascontiguousarray(lbl_patch)
            
            if random.random() > 0.5:
                img_patch = img_patch * random.uniform(0.8, 1.2)
            if random.random() > 0.5:
                img_patch = img_patch + random.uniform(-0.1, 0.1)
        
        fg_mask = (lbl_patch == 1).astype(np.float32)
        ig_mask = (lbl_patch == 2).astype(np.float32)
        
        return {
            'image': torch.from_numpy(img_patch).unsqueeze(0).float(),
            'label': torch.from_numpy(fg_mask).unsqueeze(0).float(),
            'ignore_mask': torch.from_numpy(ig_mask).unsqueeze(0).float(),
        }


print("Dataset class defined")

In [ ]:
# =============================================================================
# CELL 7: DISTRIBUTED TRAINING FUNCTION FOR TPU
# =============================================================================

def train_fn(index, cfg, train_dataset):
    """
    Training function for each TPU core.
    
    Args:
        index: TPU core index (0-7)
        cfg: Configuration
        train_dataset: Pre-loaded dataset
    """
    import time
    import torch_xla.runtime as xr  # Import in spawned process (PJRT runtime API)
    
    # Get XLA device for this process
    device = xm.xla_device()
    
    # Set seed for reproducibility
    torch.manual_seed(cfg.SEED + index)
    
    # Build model
    model = ResEncUNet3D(
        in_ch=1,
        out_ch=1,
        features=cfg.FEATURES,
        n_blocks=cfg.N_BLOCKS,
        use_scse=cfg.USE_SCSE,
        use_deep_supervision=cfg.USE_DEEP_SUPERVISION,
    ).to(device)
    
    # Broadcast parameters from master (rank 0)
    xm.broadcast_master_param(model)
    
    if xm.is_master_ordinal():
        print(f"Model parameters: {count_parameters(model):,}")
    
    # Loss function
    criterion = CombinedLoss(
        dice_weight=cfg.DICE_WEIGHT,
        bce_weight=cfg.BCE_WEIGHT,
        skeleton_weight=cfg.SKELETON_WEIGHT,
        cldice_weight=cfg.CLDICE_WEIGHT,
        skeleton_start=cfg.SKELETON_START_EPOCH,
        skeleton_warmup=cfg.SKELETON_WARMUP_EPOCHS,
        cldice_start=cfg.CLDICE_START_EPOCH,
        cldice_warmup=cfg.CLDICE_WARMUP_EPOCHS,
        ds_weights=[0.5, 0.25, 0.125] if cfg.USE_DEEP_SUPERVISION else None,
    )
    
    # Optimizer with scaled learning rate
    scaled_lr = cfg.LEARNING_RATE * cfg.BATCH_SIZE_PER_DEVICE * cfg.NUM_TPU_CORES
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=scaled_lr,
        momentum=cfg.MOMENTUM,
        weight_decay=cfg.WEIGHT_DECAY,
        nesterov=True,
    )
    
    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=lambda epoch: (1 - epoch / cfg.EPOCHS) ** 0.9
    )
    
    # Distributed sampler - USE NEW PJRT API
    train_sampler = DistributedSampler(
        train_dataset,
        num_replicas=xr.world_size(),       # NEW: was xm.xrt_world_size()
        rank=xr.global_ordinal(),            # NEW: was xm.get_ordinal()
        shuffle=True,
    )
    
    # DataLoader
    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.BATCH_SIZE_PER_DEVICE,
        sampler=train_sampler,
        num_workers=4,
        drop_last=True,
    )
    
    # Wrap with ParallelLoader for TPU
    para_loader = pl.ParallelLoader(train_loader, [device])
    
    if xm.is_master_ordinal():
        print(f"\nStarting training on {xr.world_size()} TPU cores")  # NEW: was xm.xrt_world_size()
        print(f"Samples per epoch: {len(train_dataset)}")
        print(f"Batches per epoch per device: {len(train_loader)}")
        print(f"Scaled LR: {scaled_lr}")
        print("="*60)
    
    best_loss = float('inf')
    
    # Training loop
    for epoch in range(cfg.EPOCHS):
        epoch_start = time.time()
        model.train()
        train_sampler.set_epoch(epoch)
        criterion.current_epoch = epoch
        
        epoch_loss = 0.0
        epoch_dice = 0.0
        num_batches = 0
        
        for batch in para_loader.per_device_loader(device):
            images = batch['image']
            labels = batch['label']
            ignore_mask = batch['ignore_mask']
            
            optimizer.zero_grad()
            
            output = model(images)
            loss, dice_loss = criterion(output, labels, ignore_mask)
            
            loss.backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRADIENT_CLIP)
            
            # XLA optimizer step (handles gradient sync across TPU cores)
            xm.optimizer_step(optimizer)
            
            epoch_loss += loss.item()
            epoch_dice += dice_loss
            num_batches += 1
        
        # Step scheduler
        scheduler.step()
        
        # Sync all devices
        xm.mark_step()
        
        epoch_time = time.time() - epoch_start
        avg_loss = epoch_loss / max(num_batches, 1)
        avg_dice = epoch_dice / max(num_batches, 1)
        
        # Logging (master only)
        if xm.is_master_ordinal():
            lr = scheduler.get_last_lr()[0]
            print(f"Epoch {epoch+1}/{cfg.EPOCHS} | {epoch_time:.1f}s | "
                  f"LR: {lr:.6f} | Loss: {avg_loss:.4f} | Dice: {avg_dice:.4f}")
            
            # Save checkpoint every N epochs
            if (epoch + 1) % cfg.SAVE_EVERY == 0:
                ckpt_path = f"{cfg.CHECKPOINT_DIR}/checkpoint_epoch_{epoch+1}.pt"
                xm.save(model.state_dict(), ckpt_path)
                print(f"  Saved checkpoint: {ckpt_path}")
            
            # Save best model
            if avg_loss < best_loss:
                best_loss = avg_loss
                best_path = f"{cfg.CHECKPOINT_DIR}/best_model.pt"
                xm.save(model.state_dict(), best_path)
                print(f"  >>> New best model! Loss: {avg_loss:.4f}")
    
    # Save final model
    if xm.is_master_ordinal():
        final_path = f"{cfg.CHECKPOINT_DIR}/final_model.pt"
        xm.save(model.state_dict(), final_path)
        print(f"\nTraining complete! Final model saved to {final_path}")


print("Training function defined")
print("Key XLA differences (PyTorch XLA 2.7+):")
print("  - xm.xla_device() for device")
print("  - xr.world_size() instead of xm.xrt_world_size()")
print("  - xr.global_ordinal() instead of xm.get_ordinal()")
print("  - xm.optimizer_step() instead of optimizer.step()")
print("  - xm.mark_step() to sync devices")
print("  - xm.save() instead of torch.save()")
print("  - pl.ParallelLoader for data loading")

In [ ]:
# =============================================================================
# CELL 8: RUN TRAINING
# =============================================================================

# CONFIGURATION - Modify before running
# =====================================

# For quick testing:
# cfg.DATA_FRACTION = 0.1
# cfg.EPOCHS = 10

# For full training:
cfg.DATA_FRACTION = 1.0
cfg.EPOCHS = 1200

# =====================================

print("Loading training data...")
train_dataset = VesuviusDataset(
    data_root=cfg.DATA_ROOT,
    patch_size=cfg.PATCH_SIZE,
    is_train=True,
    data_fraction=cfg.DATA_FRACTION,
    patches_per_volume=cfg.PATCHES_PER_VOLUME,
    preload=cfg.PRELOAD_ALL,
)

print(f"\nDataset size: {len(train_dataset)}")
print(f"Starting distributed training on {cfg.NUM_TPU_CORES} TPU cores...")

# Create checkpoint directory
os.makedirs(cfg.CHECKPOINT_DIR, exist_ok=True)

# Spawn training across all TPU cores
xmp.spawn(
    train_fn,
    args=(cfg, train_dataset),
    nprocs=cfg.NUM_TPU_CORES,
    start_method='fork',
)

In [ ]:
# =============================================================================
# CELL 9: CHECK TRAINING STATUS
# =============================================================================

import glob

def check_training_status():
    """Check current training status."""
    
    print("="*60)
    print("TRAINING STATUS")
    print("="*60)
    
    # Check for checkpoints
    checkpoints = sorted(glob.glob(f"{cfg.CHECKPOINT_DIR}/*.pt"))
    
    if checkpoints:
        print(f"\nCheckpoints found: {len(checkpoints)}")
        for ckpt in checkpoints[-5:]:
            print(f"  {os.path.basename(ckpt)}")
        
        # Check best model
        best_path = f"{cfg.CHECKPOINT_DIR}/best_model.pt"
        if os.path.exists(best_path):
            print(f"\nBest model: {best_path}")
    else:
        print("\nNo checkpoints found. Training not started.")
    
    print("="*60)

check_training_status()

In [ ]:
# =============================================================================
# CELL 10: INFERENCE (Single Device)
# =============================================================================

@torch.no_grad()
def sliding_window_inference(
    model,
    volume: np.ndarray,
    patch_size: Tuple[int, int, int] = (192, 192, 192),
    overlap: float = 0.5,
    device = None,
) -> np.ndarray:
    """Sliding window inference with Gaussian weighting."""
    model.eval()
    
    if device is None:
        device = xm.xla_device()
    
    D, H, W = volume.shape
    pd, ph, pw = patch_size
    sd, sh, sw = int(pd * (1 - overlap)), int(ph * (1 - overlap)), int(pw * (1 - overlap))
    
    pred_sum = np.zeros((D, H, W), dtype=np.float32)
    count = np.zeros((D, H, W), dtype=np.float32)
    
    # Gaussian weighting
    sigma = 0.125
    gauss_z = np.exp(-0.5 * ((np.arange(pd) - pd/2) / (pd * sigma)) ** 2)
    gauss_y = np.exp(-0.5 * ((np.arange(ph) - ph/2) / (ph * sigma)) ** 2)
    gauss_x = np.exp(-0.5 * ((np.arange(pw) - pw/2) / (pw * sigma)) ** 2)
    gauss_weight = gauss_z[:, None, None] * gauss_y[None, :, None] * gauss_x[None, None, :]
    
    z_pos = list(set(list(range(0, max(1, D-pd+1), sd)) + ([D-pd] if D > pd else [0])))
    y_pos = list(set(list(range(0, max(1, H-ph+1), sh)) + ([H-ph] if H > ph else [0])))
    x_pos = list(set(list(range(0, max(1, W-pw+1), sw)) + ([W-pw] if W > pw else [0])))
    
    vol_norm = (volume - volume.mean()) / (volume.std() + 1e-8)
    
    for z in tqdm(z_pos, desc="Inference", leave=False):
        for y in y_pos:
            for x in x_pos:
                patch = vol_norm[z:z+pd, y:y+ph, x:x+pw].astype(np.float32)
                actual_shape = patch.shape
                
                if patch.shape != (pd, ph, pw):
                    pad = [(0, pd-patch.shape[0]), (0, ph-patch.shape[1]), (0, pw-patch.shape[2])]
                    patch = np.pad(patch, pad, mode='reflect')
                
                inp = torch.from_numpy(patch[None, None]).to(device)
                out = model(inp)
                if isinstance(out, dict):
                    out = out['output']
                out = torch.sigmoid(out).squeeze().cpu().numpy()
                
                out = out[:actual_shape[0], :actual_shape[1], :actual_shape[2]]
                weight = gauss_weight[:actual_shape[0], :actual_shape[1], :actual_shape[2]]
                
                pred_sum[z:z+out.shape[0], y:y+out.shape[1], x:x+out.shape[2]] += out * weight
                count[z:z+out.shape[0], y:y+out.shape[1], x:x+out.shape[2]] += weight
    
    return pred_sum / np.maximum(count, 1e-8)


print("Inference function defined")